# Custom FITS: configure and run your own inversion

Upload one calibrated **Fe I 6173** Stokes cube with correct solar WCS/observer/time metadata. A continuum image or existing magnetogram is insufficient. Set the reader options below to match your actual axes, Stokes order/signs and wavelengths; [custom instruments](https://spinnme.readthedocs.io/en/stable/custom.html) covers other lines and measured responses.

The example uses a monochromatic response. No WCS, spectral calibration or instrument response is inferred from the filename.

Start with a small spatial region. For larger cubes, [crop the calibrated FITS](https://spinnme.readthedocs.io/en/stable/cropping.html) before uploading, preserving the spectral dimensions and WCS.


## 1. Install and choose a workspace

In Colab, select **Runtime → Change runtime type → GPU** if available. SPINNME detects the device automatically; CPU also works for small trials. Run cells in order. If installation requests a runtime restart, restart and continue here.

Files under `/content` are temporary. Download the results in the last step before ending your session. [Colab runtime limits](https://research.google.com/colaboratory/faq.html#resource-limits) depend on availability.


In [ ]:
%pip -q install spinnme


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

def spinnme(*arguments):
    subprocess.run([sys.executable, "-m", "spinnme", *map(str, arguments)], check=True)

ROOT = Path("/content/spinnme/custom")
ROOT.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "raw"
print("Workspace:", ROOT)
print("CUDA available:", torch.cuda.is_available())
spinnme("--version")


## 2. Upload a FITS cube and describe its layout

The default NumPy order is `(wavelength, Stokes, y, x)`. Header `WAVELNnn`, `WAVELNTH` and `WAVEUNIT` supply calibrated wavelengths. If those are absent, fill the verified `wavelength_center`/`wavelength_grid` in `READER`, with their units and offset policy. Use a two-dimensional solar spatial WCS; set `spatial_wcs_axes` if it is on different FITS axes.


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload one calibrated Stokes FITS cube.")
RAW.mkdir(parents=True, exist_ok=True)
name, contents = next(iter(uploaded.items()))
INPUT_FITS = RAW / Path(name).name
INPUT_FITS.write_bytes(contents)
del uploaded, contents
spinnme("inspect", INPUT_FITS)


In [ ]:
from spinnme.data.fits import read_fits_observation

READER = {
    "data_hdu": 0,
    "axis_order": ["wavelength", "stokes", "y", "x"],
    "stokes_order": "IQUV",
    "stokes_signs": [1, 1, 1, 1],
    "wavelength_center": None,
    "wavelength_grid": None,
    "wavelength_unit": "Angstrom",
    "wavelength_grid_is_offsets": True,
}
STOKES_NORMALIZATION = 1.0  # Divides I/Q/U/V; use your calibrated intensity scale.
# For another spatial-axis layout set, e.g., READER["spatial_wcs_axes"] = [3, 4].
observation = read_fits_observation(INPUT_FITS, **READER)
print("Canonical (y, x, Stokes, wavelength):", observation["stokes"].shape)


## 3. Save one run YAML and fit

This small configuration is a **starting trial**, not a convergence guarantee. It uses a reduced network and batch size to fit a notebook runtime. Increase the budget only after checking the spectra below. W&B is disabled; no account is needed.

For a new setup use a new `RUN_NAME`. To continue a compatible checkpoint, set `RESUME=True` and increase `MAX_STEPS`. Reuse the prepared input instead of rerunning crop/generation cells into an occupied directory.


In [ ]:
RUN_NAME = "trial-01"  # Change this for a different scientific setup.
MAX_STEPS = 1000
RESUME = False  # Set True only to resume the same run/checkpoint.
RUN = ROOT / "runs" / RUN_NAME
PRODUCTS = RUN / "products"


In [ ]:
from spinnme.configuration import resolve_config, write_config

config = resolve_config("generic/stokes_cube", output=str(RUN), input_path=str(INPUT_FITS))
for dataset in [*config["data"]["train_configs"], config["data"]["valid_config"]]:
    dataset.update(READER)
    dataset["stokes_normalization"] = STOKES_NORMALIZATION
config["data"].update(batch_size=512, dataset_batch_size=128,
                      num_workers=0, num_dataset_workers=0)
config["model"].update(dim=64, n_layers=3, encoding_config={
    "type": "fourier", "num_frequencies": [8, 8, 8, 4],
    "max_frequencies": [64, 64, 64, 4],
})
config["training"].update(max_steps=MAX_STEPS, epochs=-1,
    check_val_every_n_epoch=None, val_check_interval=max(1, min(MAX_STEPS, 250)),
    lr_params={"start": 0.001, "end": 0.0001, "iterations": 1000},
    optimizer_config={"type": "adamw"})
config["logging"].update(project="spinnme", name=RUN_NAME, mode="disabled")
CONFIG = ROOT / "configs" / f"{RUN_NAME}.yaml"
write_config(config, CONFIG)
print(CONFIG.read_text())


In [ ]:
arguments = ["run", "--config", CONFIG]
if RESUME:
    arguments.append("--resume")
spinnme(*arguments)
SNAPSHOT = RUN / "inversion.pme"


## 4. Export the fitted maps

Load the `.pme` once, then write **one FITS per parameter**, an NPZ bundle and JPG previews. `Brtp` expands to `Br`, `Btheta`, `Bphi`; `Blos` is positive toward the observer and `Vdop` is positive redshift in m/s. FITS/NPZ retain WCS and masks; JPGs are quick looks. No azimuth disambiguation is applied.

The JSON/CSV summaries describe the fitted maps, not accuracy against truth. The spectral check in the next step assesses the fit.


In [ ]:
from spinnme import load
from spinnme.evaluation import evaluate_result, write_metrics

maps, spectra = {}, {}
with load(SNAPSHOT) as snapshot:
    saved = [row for row in snapshot.observations if row["role"] == "train"]
    loss_settings = dict(snapshot.stokes_loss_config)
    for row in saved:
        identifier = row["id"]
        parameters = ["Brtp", "B", "Blos", "Vdop", "vmac", "damping"]
        if config["instrument"][0].get("artifact_correction"):
            parameters.append("Artifact")
        result = snapshot.evaluate(identifier, parameters=parameters)
        own = PRODUCTS / identifier
        result.write_fits(own / "fits", overwrite=True)
        result.write_npz(own / "parameters.npz", overwrite=True)
        result.write_jpg(own / "preview", overwrite=True)
        write_metrics(evaluate_result(result), own / "summary.json")
        maps[identifier] = result
        spectra[identifier] = snapshot.evaluate(identifier, parameters=["IQUV"])
        print(identifier, row["instrument_id"], row["shape"], row["time"]["model_time"])
print("Exports:", PRODUCTS)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.wcs import WCS

first_id = saved[0]["id"]
result = maps[first_id]
figure, axes = plt.subplots(1, 2, figsize=(10, 4),
                           subplot_kw={"projection": WCS(result.header).celestial})
for axis, key in zip(axes, ["Blos", "Vdop"]):
    values = result.arrays[key]
    limit = max(float(np.nanpercentile(np.abs(values), 99)), 1.0)
    image = axis.imshow(values, origin="lower", cmap="RdBu_r", vmin=-limit, vmax=limit)
    axis.set_title(key)
    figure.colorbar(image, ax=axis, label=result.units[key])
figure.tight_layout()
figure.savefig(PRODUCTS / "field-preview.png", dpi=150)
plt.show()


## 5. Check observed versus fitted Stokes

Inspect all four components and look for structured residuals. The table reports RMS residuals over valid pixels/wavelengths in the plotted intensity normalization, **before asinh scaling**. It is not reduced chi-squared or a parameter uncertainty. With continuum-normalized fits, each observed/synthetic profile uses its own intensity quantile. Compare residuals with noise estimated in the same normalization.

This is a fit check on training data. Use held-out acquisitions to assess interpolation; one acquisition does not establish temporal evolution.


In [ ]:
from spinnme.data.fits import read_fits_observation

def read_observed(row):
    return read_fits_observation(row["source_paths"][0], **READER)["stokes"]


In [ ]:
import json

rms_rows = []
for row in saved:
    identifier = row["id"]
    fitted = spectra[identifier]
    measured = read_observed(row) / row["stokes_normalization"]
    if loss_settings.get("continuum_normalization", False):
        quantile = loss_settings.get("continuum_quantile", 0.9)
        continuum = np.quantile(measured[..., 0:1, :], quantile, axis=-1, keepdims=True)
        measured = measured / np.where(continuum > 0, continuum, np.nan)
    predicted = np.stack([fitted.arrays[key] for key in "IQUV"], axis=-2)
    if measured.shape != predicted.shape:
        raise ValueError("Observed and fitted grids/wavelength selections must match.")
    valid = fitted.mask & np.isfinite(measured).all(axis=(-2, -1))
    if not valid.any() or not np.isfinite(predicted[valid]).all():
        raise ValueError("No valid comparison or non-finite fitted spectra.")
    residual = predicted - measured
    rms = np.sqrt(np.mean(residual[valid] ** 2, axis=(0, 2)))
    rms_rows.append({"observation": identifier, "instrument": row["instrument_id"],
                     **dict(zip("IQUV", map(float, rms)))})
    print(identifier, "I/Q/U/V RMS:", rms)
    y, x = np.argwhere(valid)[len(np.argwhere(valid)) // 2]
    wavelength = (np.asarray(fitted.metadata["wavelength_grid_m"])
                  + fitted.metadata["wavelength_center_m"]) * 1e10
    figure, axes = plt.subplots(2, 4, figsize=(12, 5), sharex=True)
    for index, component in enumerate("IQUV"):
        axes[0, index].plot(wavelength, measured[y, x, index], "o", label="observed")
        axes[0, index].plot(wavelength, predicted[y, x, index], label="fitted")
        axes[0, index].set_title(component)
        axes[1, index].plot(wavelength, residual[y, x, index])
        axes[1, index].axhline(0, color="k", linewidth=0.5)
        axes[1, index].set_xlabel("Wavelength [Å]")
    axes[0, 0].set_ylabel("Stokes intensity")
    axes[1, 0].set_ylabel("Fitted − observed")
    axes[0, 0].legend()
    figure.suptitle(identifier)
    figure.tight_layout()
    figure.savefig(PRODUCTS / identifier / "stokes-fit.png", dpi=150)
    plt.show()
(PRODUCTS / "stokes-rms.json").write_text(json.dumps(rms_rows, indent=2))


## 6. Save your results

Download the snapshot, run YAML/checkpoint, metrics and exported products. Input observations, calibration files and the large preparation cache are excluded. A `.pme` can export its saved grids without the original observations; resuming training also needs the same inputs/calibration. Keep those separately if you plan to resume after the Colab runtime ends.


In [ ]:
import shutil
from google.colab import files

bundle = ROOT / "download" / RUN_NAME
bundle.mkdir(parents=True, exist_ok=True)
for name in ["inversion.pme", "last.ckpt", "resolved.yaml", "run_manifest.json"]:
    shutil.copy2(RUN / name, bundle / name)
shutil.copy2(CONFIG, bundle / "run.yaml")
shutil.copytree(PRODUCTS, bundle / "products", dirs_exist_ok=True)
archive = shutil.make_archive(str(ROOT / RUN_NAME), "zip", bundle)
files.download(archive)


Next: [training options](https://spinnme.readthedocs.io/en/stable/training_options.html), [fit quality](https://spinnme.readthedocs.io/en/stable/evaluation.html), [Python exports at custom resolution/cadence](https://spinnme.readthedocs.io/en/stable/python_exports.html), or the [beginner FAQ](https://spinnme.readthedocs.io/en/stable/faq.html).
